# W1D3: 히스토그램 분석

> 📖 원리 이해: [OpenCV 공식 튜토리얼 - Histograms: Find, Plot, Analyze](https://docs.opencv.org/4.x/d1/db7/tutorial_py_histogram_begins.html)
> 📋 파라미터 확인: [OpenCV 공식 - calcHist](https://docs.opencv.org/4.x/d6/dc7/group__imgproc__hist.html#ga4b2b8fd75503ff9d6f3d5e4e9aa3dbcc)

## 📌 오늘의 목표
- 히스토그램으로 이미지의 밝기 분포를 수치로 파악하기
- 결함 유형별 히스토그램 패턴이 다름을 확인하기
- 정규화 히스토그램과 누적 히스토그램(CDF) 이해하기

## 🎯 배울 함수

### `cv2.calcHist(images, channels, mask, histSize, ranges)`
| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `images` | `[img]` | 입력 이미지 리스트. 반드시 `[]`로 감싸야 함 |
| `channels` | `[0]` | 계산할 채널 번호. Grayscale=`[0]`, BGR B채널=`[0]`, G=`[1]`, R=`[2]` |
| `mask` | `None` or `np.array` | `None`이면 전체 이미지. ROI만 계산하려면 0/255 마스크 배열 전달 |
| `histSize` | `[256]` | bin(구간) 개수. `[256]`이면 0~255 각 밝기값마다 1칸 |
| `ranges` | `[0, 256]` | 픽셀 값 범위. 끝값 256은 포함 안 됨 (0~255 포함) |

### `cv2.normalize(src, dst, alpha, beta, norm_type)`
| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `src` | array | 정규화할 원본 배열 (히스토그램) |
| `dst` | `None` or array | 결과 저장 배열. `None`이면 새로 생성 |
| `alpha` | `0` | 정규화 후 최솟값 |
| `beta` | `1.0` | 정규화 후 최댓값 |
| `norm_type` | `cv2.NORM_MINMAX` | 정규화 방식. `NORM_MINMAX`=최소~최대 범위로 스케일 |

### `np.cumsum(a)`
| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `a` | array | 누적합을 구할 배열 (정규화된 히스토그램) |
| 반환값 | array | 왼쪽부터 누적합한 배열. 마지막 값 = 전체 합 |

## 📦 import + 데이터 로딩

히스토그램 분석을 위해 grayscale로 로딩합니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path("../data/kaggle/archive/NEU-DET/train/images")
DEFECT_TYPES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

img = cv2.imread(str(DATA_DIR / "inclusion" / "inclusion_1.jpg"), cv2.IMREAD_GRAYSCALE)
print(f"로딩 완료: shape={img.shape}, dtype={img.dtype}")
print(f"픽셀 범위: {img.min()} ~ {img.max()}, 평균: {img.mean():.1f}")

## 🔧 1. calcHist 기본 사용법

**`cv2.calcHist([img], [channel], mask, [histSize], [ranges])`** 로 히스토그램을 계산합니다.
- `[img]`: 입력 이미지 (리스트로 감싸야 함)
- `[channel]`: 계산할 채널 번호. Grayscale이면 `[0]`
- `mask`: 특정 영역만 계산할 때 사용. 없으면 `None` (전체)
- `[histSize]`: 구간(bin) 개수. `[256]`이면 0~255 각각 1칸씩
- `[ranges]`: 픽셀 값 범위. `[0, 256]`이 기본값

**제조 검사 연결:** 히스토그램은 이미지의 "밝기 지문"입니다. 정상 제품과 불량 제품의 히스토그램 패턴이 다르면, 히스토그램만으로도 양/불 판별이 가능합니다.

In [ ]:
hist = cv2.calcHist([img], [0], None, [256], [0, 256])

print(f"hist shape: {hist.shape}  → (256개 bin, 1열)")
print(f"hist dtype: {hist.dtype}")
print(f"총 픽셀 수 합산: {hist.sum():.0f}  (= {img.shape[0]} x {img.shape[1]} = {img.size})")
print(f"가장 많은 밝기값: {hist.argmax()} (픽셀 수: {hist.max():.0f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title("inclusion_1 (Grayscale)", fontsize=13)
axes[0].axis('off')

axes[1].plot(hist, color='steelblue', linewidth=1.5)
axes[1].fill_between(range(256), hist.flatten(), alpha=0.3, color='steelblue')
axes[1].set_title("히스토그램 (calcHist)", fontsize=13)
axes[1].set_xlabel("픽셀 밝기 (0=검정, 255=흰색)", fontsize=11)
axes[1].set_ylabel("픽셀 수", fontsize=11)
axes[1].set_xlim([0, 256])
axes[1].axvline(x=hist.argmax(), color='red', linestyle='--', alpha=0.7,
                label=f"최빈값: {hist.argmax()}")
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.suptitle("inclusion_1: 이미지 + 히스토그램", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- X축(밝기)에서 **봉우리 위치** = 이미지의 주된 밝기
- **봉우리가 왼쪽** = 어두운 이미지, **오른쪽** = 밝은 이미지, **가운데** = 중간 밝기
- **봉우리가 뾰족하고 좁음** = 밝기가 균일한 표면, **넓게 퍼짐** = 밝기 다양한 텍스처
- 결함 부위는 주변과 밝기가 다르므로 히스토그램에 **작은 봉우리(꼬리)**로 나타날 수 있음

> 🤔 **판단 질문:** 히스토그램에서 봉우리가 두 개(쌍봉) 보인다면 이미지에서 무엇이 있다는 뜻일까요?

## 🔧 2. 6종 결함 히스토그램 비교

결함 유형마다 표면 텍스처가 다르므로 히스토그램 모양도 달라집니다.

**제조 검사 연결:** 결함 유형을 미리 분류한 데이터가 없을 때, 히스토그램 패턴으로 결함을 군집화(clustering)하는 접근이 가능합니다.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(24, 9))

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']

for col, (defect, color) in enumerate(zip(DEFECT_TYPES, colors)):
    img_d = cv2.imread(str(DATA_DIR / defect / f"{defect}_1.jpg"), cv2.IMREAD_GRAYSCALE)
    hist_d = cv2.calcHist([img_d], [0], None, [256], [0, 256])

    axes[0, col].imshow(img_d, cmap='gray')
    axes[0, col].set_title(defect.replace('_', '\n'), fontsize=11, fontweight='bold')
    axes[0, col].axis('off')
    mean_v = img_d.mean()
    std_v  = img_d.std()
    axes[0, col].set_xlabel(f"mean={mean_v:.0f}\nstd={std_v:.1f}", fontsize=9)

    axes[1, col].plot(hist_d, color=color, linewidth=1.5)
    axes[1, col].fill_between(range(256), hist_d.flatten(), alpha=0.25, color=color)
    axes[1, col].set_xlim([0, 256])
    axes[1, col].set_xlabel("밝기 (0~255)", fontsize=9)
    axes[1, col].set_ylabel("픽셀 수", fontsize=9)
    axes[1, col].axvline(x=hist_d.argmax(), color='black', linestyle='--',
                         alpha=0.5, linewidth=1)
    axes[1, col].grid(alpha=0.3)

plt.suptitle("6종 결함: 이미지 + 히스토그램 비교", fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **crazing**: 미세 균열 → 중간 밝기에 넓게 분포
- **inclusion**: 이물질 포함 → 어두운 점이 있어 왼쪽에 꼬리
- **patches**: 면 결함 → 어두운 영역이 커서 낮은 밝기 쪽에 봉우리
- **pitted_surface**: 표면 움푹 파임 → 어두운 구멍 → 이중 봉우리 가능
- **rolled-in_scale**: 이물질 압착 → 밝기 패턴이 복잡
- **scratches**: 선형 스크래치 → 어두운 선 → 낮은 밝기에 작은 피크

> 🤔 **판단 질문:** std(표준편차)가 가장 높은 결함 유형은? std가 높다는 것은 이미지에서 무엇을 의미하나요?

## 🔧 3. 정규화 히스토그램

**`cv2.normalize(hist, hist, 0, 1.0, cv2.NORM_MINMAX)`** 로 히스토그램을 0~1 범위로 스케일합니다.
- 이미지 크기가 달라도 공정하게 비교 가능
- Y축이 "픽셀 수"에서 "비율"로 바뀜 (전체 픽셀 중 몇 %)

**제조 검사 연결:** 카메라 해상도가 다른 라인끼리 비교하거나, ROI 크기가 달라도 히스토그램 모양 자체를 비교할 수 있습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for defect, color in zip(["inclusion", "scratches", "patches"], ['#3498db', '#e74c3c', '#2ecc71']):
    img_d = cv2.imread(str(DATA_DIR / defect / f"{defect}_1.jpg"), cv2.IMREAD_GRAYSCALE)
    hist_d = cv2.calcHist([img_d], [0], None, [256], [0, 256])

    axes[0].plot(hist_d, color=color, linewidth=1.5, label=defect)

    hist_norm = cv2.normalize(hist_d, None, 0, 1.0, cv2.NORM_MINMAX)
    axes[1].plot(hist_norm, color=color, linewidth=1.5, label=defect)

axes[0].set_title("원본 히스토그램 (픽셀 수)", fontsize=13)
axes[0].set_xlabel("밝기 (0~255)", fontsize=11)
axes[0].set_ylabel("픽셀 수", fontsize=11)
axes[0].set_xlim([0, 256])
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

axes[1].set_title("정규화 히스토그램 (0~1 스케일)", fontsize=13)
axes[1].set_xlabel("밝기 (0~255)", fontsize=11)
axes[1].set_ylabel("비율 (0~1)", fontsize=11)
axes[1].set_xlim([0, 256])
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.suptitle("원본 vs 정규화 히스토그램 비교", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **원본**: 이미지 크기가 같으면 비교 가능. Y축 절대값(픽셀 수)이 의미 있음
- **정규화**: Y축 범위가 통일됨 → **형태(패턴)** 비교에 적합
- 정규화해도 각 결함의 봉우리 위치와 폭은 그대로 → 결함 구별 정보 보존

> 🧪 **실험해보기:** inclusion_10.jpg, inclusion_20.jpg 등 같은 유형의 다른 이미지를 정규화 비교해보세요. 같은 결함끼리는 히스토그램이 비슷한가요?

## 🔧 4. 누적 히스토그램 (CDF)

**`np.cumsum(hist)`** 로 누적합(Cumulative Distribution Function)을 구합니다.
- 히스토그램을 왼쪽부터 누적 합산
- "밝기 X 이하인 픽셀이 전체의 몇 %인가"를 한눈에 파악
- W1D4에서 배울 `equalizeHist`(히스토그램 평활화)의 핵심 원리

**제조 검사 연결:** CDF에서 급격한 계단이 있다면 해당 밝기값 근처에 픽셀이 몰려있다는 뜻 → 대비가 낮은 이미지임을 진단할 수 있습니다.

In [ ]:
img_inc = cv2.imread(str(DATA_DIR / "inclusion" / "inclusion_1.jpg"), cv2.IMREAD_GRAYSCALE)
hist_inc = cv2.calcHist([img_inc], [0], None, [256], [0, 256])
hist_norm = cv2.normalize(hist_inc, None, 0, 1.0, cv2.NORM_MINMAX).flatten()
cdf = np.cumsum(hist_norm)
cdf_scaled = cdf / cdf.max()

print(f"밝기 128 이하 픽셀 비율: {cdf_scaled[128]:.1%}")
print(f"밝기 200 이하 픽셀 비율: {cdf_scaled[200]:.1%}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img_inc, cmap='gray')
axes[0].set_title("inclusion_1", fontsize=13)
axes[0].axis('off')

axes[1].plot(hist_norm, color='steelblue', linewidth=1.5)
axes[1].fill_between(range(256), hist_norm, alpha=0.3, color='steelblue')
axes[1].set_title("정규화 히스토그램 (PDF)", fontsize=13)
axes[1].set_xlabel("밝기 (0~255)", fontsize=11)
axes[1].set_xlim([0, 256])
axes[1].grid(alpha=0.3)

axes[2].plot(cdf_scaled, color='darkorange', linewidth=2)
axes[2].axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='50%')
axes[2].axvline(x=128, color='gray', linestyle=':', alpha=0.7, label='밝기 128')
axes[2].scatter([128], [cdf_scaled[128]], color='red', zorder=5,
                label=f'128 이하: {cdf_scaled[128]:.1%}')
axes[2].set_title("누적 히스토그램 (CDF)", fontsize=13)
axes[2].set_xlabel("밝기 (0~255)", fontsize=11)
axes[2].set_ylabel("누적 비율 (0~1)", fontsize=11)
axes[2].set_xlim([0, 256])
axes[2].set_ylim([0, 1.05])
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)

plt.suptitle("inclusion_1: PDF(히스토그램) vs CDF(누적)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- CDF가 급격히 오르는 구간 = 그 밝기 근처에 픽셀이 집중된 것
- CDF가 완만한 구간 = 그 밝기값의 픽셀이 거의 없음
- **이상적인 이미지**: CDF가 0~255 전체에 걸쳐 직선에 가까움 (대비 풍부)
- **대비 낮은 이미지**: CDF가 좁은 범위에서 급격히 오름 → `equalizeHist`로 개선 (W1D4)

> 🤔 **판단 질문:** CDF 그래프에서 '계단'이 두 군데 뚜렷하게 보인다면 이미지에서 무슨 일이 일어나고 있는 걸까요?

## 🔧 5. ROI 마스크 히스토그램 — 결함 영역만 분석

**mask 파라미터**를 사용하면 전체가 아닌 특정 영역의 히스토그램만 계산할 수 있습니다.
- mask는 0(제외) 또는 255(포함)인 같은 크기의 uint8 배열

**제조 검사 연결:** 검사 영역(ROI)과 배경을 분리해서, 결함 부위의 밝기 분포만 추출합니다. 전체 이미지 히스토그램은 배경에 희석되므로, 마스크 히스토그램이 더 정확한 결함 정보를 줍니다.

In [ ]:
img_patch = cv2.imread(str(DATA_DIR / "patches" / "patches_1.jpg"), cv2.IMREAD_GRAYSCALE)
h, w = img_patch.shape

mask_roi = np.zeros((h, w), dtype=np.uint8)
mask_roi[h//4 : 3*h//4, w//4 : 3*w//4] = 255

hist_full = cv2.calcHist([img_patch], [0], None,   [256], [0, 256])
hist_roi  = cv2.calcHist([img_patch], [0], mask_roi, [256], [0, 256])

hist_full_n = cv2.normalize(hist_full, None, 0, 1.0, cv2.NORM_MINMAX).flatten()
hist_roi_n  = cv2.normalize(hist_roi,  None, 0, 1.0, cv2.NORM_MINMAX).flatten()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

img_vis = cv2.cvtColor(img_patch, cv2.COLOR_GRAY2RGB)
cv2.rectangle(img_vis, (w//4, h//4), (3*w//4, 3*h//4), (255, 0, 0), 2)
axes[0].imshow(img_vis)
axes[0].set_title("patches_1 (파란 박스 = ROI)", fontsize=13)
axes[0].axis('off')

axes[1].imshow(mask_roi, cmap='gray')
axes[1].set_title("마스크 (흰색 영역만 계산)", fontsize=13)
axes[1].axis('off')

axes[2].plot(hist_full_n, color='steelblue', linewidth=1.5, label='전체 이미지', alpha=0.8)
axes[2].plot(hist_roi_n,  color='darkorange', linewidth=1.5, label='ROI 마스크', alpha=0.8)
axes[2].set_title("전체 vs ROI 마스크 히스토그램", fontsize=13)
axes[2].set_xlabel("밝기 (0~255)", fontsize=11)
axes[2].set_ylabel("정규화 값", fontsize=11)
axes[2].set_xlim([0, 256])
axes[2].legend(fontsize=11)
axes[2].grid(alpha=0.3)

plt.suptitle("마스크 히스토그램: 전체 vs ROI 비교", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- 두 히스토그램의 모양이 **비슷하면** = ROI가 전체를 잘 대표하는 영역
- **다르면** = ROI에 특별한 패턴(결함 집중 또는 배경 제외)이 있다는 뜻
- mask를 None 대신 배열로 넘기면 그 영역 픽셀만 집계됨

> 🧪 **실험해보기:** 마스크 영역을 `[0:h//3, :]` (상단 1/3)로 바꿔보세요. 결함이 이미지 어디에 집중되어 있는지 히스토그램으로 찾아낼 수 있을까요?

## 🔬 파라미터 실험: bins 수 변경

bins 수를 바꾸면 히스토그램의 세밀함이 달라집니다.
- bins가 많을수록 세밀하지만 노이즈에 민감
- bins가 적을수록 부드럽지만 세부 정보 손실

In [ ]:
img_test = cv2.imread(str(DATA_DIR / "pitted_surface" / "pitted_surface_1.jpg"), cv2.IMREAD_GRAYSCALE)

bins_list = [8, 32, 64, 256]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, bins in zip(axes, bins_list):
    hist_b = cv2.calcHist([img_test], [0], None, [bins], [0, 256])
    hist_b_n = cv2.normalize(hist_b, None, 0, 1.0, cv2.NORM_MINMAX).flatten()
    x = np.linspace(0, 255, bins)
    ax.bar(x, hist_b_n, width=256/bins, color='steelblue', alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.set_title(f"bins={bins}", fontsize=13, fontweight='bold')
    ax.set_xlabel("밝기 (0~255)", fontsize=10)
    ax.set_ylabel("정규화 값", fontsize=10)
    ax.set_xlim([0, 256])
    ax.grid(alpha=0.3)

plt.suptitle("pitted_surface_1: bins 수에 따른 히스토그램 변화", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **bins=8**: 큰 덩어리로만 분포 파악 → 세밀한 차이 놓침
- **bins=32**: 검사 자동화에서 자주 쓰는 균형점
- **bins=64**: 결함 유형 분류 비교에 적합
- **bins=256**: 가장 세밀 → 노이즈에 민감, 시각화 복잡

> 🧪 **실험해보기:** bins=16으로 6종 결함을 비교해보세요. bins=256보다 유형 간 차이가 더 잘 보이는 유형이 있나요?

> **실무 요약**: 히스토그램 비교(분류)에는 bins=32~64, 시각화·디버깅에는 bins=256

## 📝 정리

### 오늘 배운 것
| 함수/개념 | 핵심 포인트 |
|-----------|-------------|
| `cv2.calcHist()` | 입력은 `[img]` 리스트, channel `[0]`, mask `None`, bins `[256]`, range `[0,256]` |
| `cv2.normalize()` | 히스토그램을 0~1 범위로 스케일 → 크기 다른 이미지 공정 비교 |
| `np.cumsum(hist)` | 누적 히스토그램(CDF) — "밝기 X 이하 픽셀 비율" 한눈에 파악 |
| mask 파라미터 | 특정 ROI만의 히스토그램 추출 — 결함 영역 집중 분석 |
| bins 수 | 많을수록 세밀하지만 노이즈 민감. 비교용=32~64, 시각화=256 |

### 핵심 개념
- 히스토그램 = 이미지의 **밝기 지문** → 결함 유형마다 패턴이 다름
- **봉우리 위치**: 이미지의 주 밝기 / **봉우리 폭**: 밝기 균일성
- **CDF**: 히스토그램의 누적값 → 다음 시간(D4)의 equalizeHist 원리
- **mask**: ROI 히스토그램 추출 → 배경 제거 후 결함 부위만 정밀 분석

### 복습 퀴즈
1. `cv2.calcHist([img], [0], None, [256], [0, 256])`에서 숫자 `256`이 두 번 나오는데, 각각의 의미는?
2. 히스토그램 봉우리가 왼쪽 끝(밝기 0~30)에 몰려있다면 이 이미지는 어떤 상태인가?
3. CDF(누적 히스토그램)에서 특정 구간의 기울기가 가파르다면 어떤 의미인가?
4. bins=8과 bins=256 중 결함 유형 분류에 더 적합한 것은? 이유는?

### 다음에 할 것 (W1D4)
- 밝기·대비 조절 — `equalizeHist`, `createCLAHE`
- 오늘 배운 히스토그램이 평평해지도록 자동 조절하는 알고리즘